In [1]:
# ============================================================
# M5 TIER-2 CEILING PIN AUDITOR
# Exact reduction: prove B_L <= 57/1210 => T_C < 1/8 => N*_C >= 8
# Self-contained. No files required.
# ============================================================

import math, json, numpy as np

# ----------------------------
# AF diagonal constants
# ----------------------------
BETA0 = 5.6
GAMMA = 11.0 / (8.0 * math.pi**2)      # d beta / d ln s
A = 3.0 / (32.0 * math.pi**2)          # analytic log-slope of Y = alpha_W^2 T_C
BCRIT = 57.0 / 1210.0                  # exact threshold for T_C < 1/8

print("=" * 88)
print("M5 TIER-2 CEILING PIN AUDITOR")
print("=" * 88)
print(f"A      = 3/(32π²)     = {A:.18f}")
print(f"gamma  = 11/(8π²)     = {GAMMA:.18f}")
print(f"A/gamma              = {A/GAMMA:.18f}  expected 3/44 = {3/44:.18f}")
print(f"Bcrit = 57/1210      = {BCRIT:.18f}")
print()

# ----------------------------
# Exact closed-form lattice sums
# ----------------------------
def beta_of_s(s):
    return BETA0 + GAMMA * math.log(s)

def m02_of_s(s):
    return 0.5 / (s * s)

def reduced_1d(L):
    ns = np.arange(L // 2 + 1)
    vals = 4.0 * np.sin(np.pi * ns / L) ** 2
    mult = np.full(len(ns), 2.0)
    mult[0] = 1.0
    if L % 2 == 0:
        mult[-1] = 1.0
    return vals, mult

def sums12(L, m02, aW, chunk_elems=6_000_000):
    """
    Computes S1, S2 over all n in {0,...,L-1}^4, including n=0:
        S1 = sum (m02 + aW*w(n))^-1
        S2 = sum (m02 + aW*w(n))^-2
    Uses reduced 1D multiplicities.
    """
    v, c = reduced_1d(L)
    V2 = (v[:, None] + v[None, :]).ravel()
    C2 = (c[:, None] * c[None, :]).ravel()

    nB = len(V2)
    step = max(1, chunk_elems // nB)

    S1 = 0.0
    S2 = 0.0

    for i in range(0, nB, step):
        denom = m02 + aW * (V2[i:i+step, None] + V2[None, :])
        R = 1.0 / denom
        W = C2[i:i+step, None] * C2[None, :]
        WR = W * R
        S1 += float(np.sum(WR))
        S2 += float(np.sum(WR * R))

    return S1, S2

def consts(L, m02, aW):
    Ns = float(L) ** 4
    S1, S2 = sums12(L, m02, aW)

    # remove zero mode from coexact sum
    S1 -= 1.0 / m02
    S2 -= 1.0 / (m02 * m02)

    g_H = 1.0 / (m02 * Ns)
    T_H = 1.0 / (m02 * m02 * Ns)

    g_C = (3.0 / (4.0 * Ns)) * S1
    T_C = (3.0 / (4.0 * Ns)) * S2

    return {
        "g_H": g_H,
        "T_H": T_H,
        "g_C": g_C,
        "T_C": T_C,
        "g_diag": g_H + g_C,
        "T_full": T_H + T_C,
    }

def nstar_split(g_H, T_C):
    best = 0
    cap = int(1.0 / T_C) + 5
    for D in range(1, cap + 1):
        if D * g_H + math.sqrt(D * T_C) < 1.0:
            best = D
        else:
            break
    return best

def radii(cs):
    return {
        "Nstar": int(1.0 // cs["T_full"]),
        "Nstar_C": int(1.0 // cs["T_C"]),
        "Nstar_split": nstar_split(cs["g_H"], cs["T_C"]),
    }

# Same diagonal used in the M4 deposit
S_LIST = [1, 1.25, 1.5, 2, 3, 4, 5, 6, 8, 10, 12, 16, 20, 24, 32, 48, 64]

rows = []
for s in S_LIST:
    L = int(round(4 * s))
    beta = beta_of_s(s)
    m02 = m02_of_s(s)
    aW = beta / 6.0
    cs = consts(L, m02, aW)
    rr = radii(cs)

    x = math.log(s)
    Y = cs["T_C"] * aW * aW
    B_L = Y - A * x
    margin = BCRIT - B_L

    row = {
        "L": L,
        "s": s,
        "beta": beta,
        "T_C": cs["T_C"],
        "T_full": cs["T_full"],
        "Y": Y,
        "B_L": B_L,
        "Bcrit_margin": margin,
        **rr
    }
    rows.append(row)

# ----------------------------
# Print table
# ----------------------------
print("FINITE DIAGONAL CHECK")
print("-" * 88)
print(f"{'L':>5} {'s':>8} {'beta':>10} {'T_C':>13} {'Y=aW^2*T_C':>16} {'B_L':>16} {'margin':>16} {'N*_C':>6}")
for r in rows:
    print(
        f"{r['L']:5d} "
        f"{r['s']:8.2f} "
        f"{r['beta']:10.6f} "
        f"{r['T_C']:13.9f} "
        f"{r['Y']:16.12f} "
        f"{r['B_L']:16.12f} "
        f"{r['Bcrit_margin']:16.12f} "
        f"{r['Nstar_C']:6d}"
    )

maxB = max(r["B_L"] for r in rows)
maxB_row = max(rows, key=lambda r: r["B_L"])
min_margin = min(r["Bcrit_margin"] for r in rows)

print()
print("B-WINDOW SUMMARY")
print("-" * 88)
print(f"max observed B_L      = {maxB:.18f} at L={maxB_row['L']}, s={maxB_row['s']}")
print(f"Bcrit                 = {BCRIT:.18f}")
print(f"minimum observed gap  = {min_margin:.18f}")
print()

# ----------------------------
# Exact ceiling implication
# ----------------------------
def ceiling_from_B(B):
    """
    Upper ceiling for T_C if Y <= A*x + B.
    """
    xstar = BETA0 / GAMMA - 2.0 * B / A
    if xstar < 0:
        xstar = 0.0
    beta_star = BETA0 + GAMMA * xstar
    Tbar = 36.0 * (A * xstar + B) / (beta_star * beta_star)
    return xstar, beta_star, Tbar

for label, B in [
    ("observed max B_L", maxB),
    ("safe B = 0.025", 0.025),
    ("safe B = 0.030", 0.030),
    ("safe B = 0.040", 0.040),
    ("critical Bcrit", BCRIT),
]:
    xstar, beta_star, Tbar = ceiling_from_B(B)
    print(f"{label:18s}: B={B:.12f}, x*={xstar:.6f}, beta*={beta_star:.6f}, Tbar={Tbar:.12f}, floor(1/Tbar)={int(1.0 // Tbar)}")

print()
print("THEOREM REDUCTION")
print("-" * 88)
print("If one proves, for every integer L >= 4 on the AF diagonal,")
print()
print("    Y_L = alpha_W(L)^2 * T_C(L) <= (3/(32*pi^2))*log(L/4) + 57/1210,")
print()
print("then automatically")
print()
print("    T_C(L) < 1/8,")
print("    N*_C(L) >= 8,")
print("    T_full(L) < 1/64 + 1/8 = 9/64,")
print("    N*(L) >= 7.")
print()
print("Observed data are far below the required B-window; the remaining task is now")
print("a scalar heat-kernel/theta-tail upper bound, not another lattice solve.")
print("=" * 88)

M5 TIER-2 CEILING PIN AUDITOR
A      = 3/(32π²)     = 0.009498860966469166
gamma  = 11/(8π²)     = 0.139316627508214441
A/gamma              = 0.068181818181818177  expected 3/44 = 0.068181818181818177
Bcrit = 57/1210      = 0.047107438016528926

FINITE DIAGONAL CHECK
----------------------------------------------------------------------------------------
    L        s       beta           T_C       Y=aW^2*T_C              B_L           margin   N*_C
    4     1.00   5.600000   0.018837704   0.016409733262   0.016409733262   0.030697704754     53
    5     1.25   5.631088   0.021491037   0.018929513065   0.016809903495   0.030297534521     46
    6     1.50   5.656488   0.023582815   0.020959788397   0.017108331708   0.029999106308     42
    8     2.00   5.696567   0.026705736   0.024072902796   0.017488794098   0.029618643918     37
   12     3.00   5.753055   0.030757138   0.028277464892   0.017841899506   0.029265538511     32
   16     4.00   5.793134   0.033430312   0.0311648509